# SDR variant dataset

Builds the annotated variant table of the SDR-seq experiments SDR007AB and SDR010ABC that is used
as VEP input and for all SDR analyses:

1. Load the MAST and Wilcoxon differential expression results per experiment. A variant-gene pair is
   **positive** if `p_val_adj <= 0.05` and **negative** if `p_val_adj == 1`, at least 100 cells carry the
   variant and `|avg_log2FC| <= 0.1`; both tests must agree. Replicates (SDR007A/B, SDR010A/B/C) are
   averaged, CM2/FB2 are merged into CM/FB, and the experiments are combined.
2. Convert the variants to VCF-style rows against the reference genome (`DE_to_VCF`).
3. Annotate the CM and FB variants: ENCODE4 cCREs, phyloP, GTF features and coding effects, distance to
   the TSS of the target gene, CADD (SNVs from the CADD BigWigs, indels/MNVs from the CADD website, see
   below), cardioid ABC enhancers and overlap with the pseudobulk ATAC peaks. Days and phased duplicates
   are merged; variant-gene pairs with contradictory labels are removed.

**Outputs** (in `variant_data_dir`; the writing lines are commented out):
`VCF_SDR007AB_DLI_SDR010ABC_Comb.csv` (VEP input) and `VCF_SDR007AB_DLI_SDR010ABC_Comb_Meta.csv`
(annotated CM/FB variant-gene pairs, read by `sdr_cohort_overview.ipynb` and
`benchmarks/sdr_vep_benchmark.ipynb`).

**Paths** come from `config/paths.yaml` (`sdr007_de_dir`, `sdr010_de_dir`, `variant_data_dir` with the
annotation tracks, `atac_peak_dir`, `abc_enhancer_table`, `genome_fasta`, `gtf`).

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import pyBigWig
from pyfaidx import Fasta

from cardiomics_s2f.config import load_paths
from cardiomics_s2f.eval.track_prediction import get_gtf
from cardiomics_s2f.io.variant_annotation import (
    annotate_cadd_snv_scores,
    annotate_cardioid_enhancer_overlap,
    annotate_cCREs,
    annotate_coding_effect,
    annotate_gtf_features,
    annotate_phyloP_score,
    celltype_from_peak_file,
    flag_variants_in_peaks,
    load_ccre_data,
    read_narrowpeak,
    to_vcf_column_names,
    vcf_to_pr,
)
from cardiomics_s2f.vep.processing import DE_to_VCF

In [ ]:
paths = load_paths()
VARIANT_DIR = paths.variant_data_dir      # annotation tracks and the VCFs written below
ANNOT_PATH = f"{VARIANT_DIR}/"            # cCREs, phyloP, CADD BigWigs and CADD indel results
SDR007_FILE_PATH = paths.sdr007_de_dir
SDR010_FILE_PATH = paths.sdr010_de_dir
VCF_OUT_PATH = VARIANT_DIR / "VCF_SDR007AB_DLI_SDR010ABC_Comb.csv"
SDR_VCF_META_FILE = VARIANT_DIR / "VCF_SDR007AB_DLI_SDR010ABC_Comb_Meta.csv"
ENHANCER_DIR = paths.abc_enhancer_table   # ABC candidate enhancers of the cardioids
PEAK_DIR = paths.atac_peak_dir            # pseudobulk ATAC narrowPeak files

hg38 = Fasta(str(paths.genome_fasta))
gtf = get_gtf(paths.gtf, chrom_filter="autosomesXY", gene_type_filter=None, feature=None)
gtf.loc[gtf.gene_name == "FAM69B", "gene_name"] =  "DIPK1B" #one of the sdr genes that has a differnt alias in the gtf

## SDR VCF file prep

In [ ]:
#Loading SDR-seq varaints
#sdr7
sdr7_mast_a = pd.read_csv(SDR007_FILE_PATH / "SDR007A_phased_MAST_cell_annot_day_complement.csv")
sdr7_wil_a = pd.read_csv(SDR007_FILE_PATH / "SDR007A_phased_Wilcox_cell_annot_day_complement.csv")

sdr7_mast_b = pd.read_csv(SDR007_FILE_PATH / "SDR007B_phased_MAST_cell_annot_day_complement.csv")
sdr7_wil_b = pd.read_csv(SDR007_FILE_PATH / "SDR007B_phased_Wilcox_cell_annot_day_complement.csv")

sdr7_mast = pd.concat([sdr7_mast_a, sdr7_mast_b])
sdr7_mast = sdr7_mast[sdr7_mast.Gene != "DUMMY-FEATURE-FOR-SEURAT"]

sdr7_mast_pos = sdr7_mast[sdr7_mast.p_val_adj <= 0.05]
sdr7_mast_neg = sdr7_mast[(sdr7_mast.p_val_adj == 1) & (sdr7_mast.Ncells_Var1 >= 100) & (sdr7_mast.avg_log2FC.abs() <= 0.1)]

sdr7_mast_pos.loc[:, "var_type"] = "positive"
sdr7_mast_neg.loc[:,"var_type"] = "negative"

sdr7_mast = pd.concat([sdr7_mast_pos, sdr7_mast_neg])
sdr7_mast = sdr7_mast.groupby( #average sdr7a and b replicates
    ["ID_Geno_Var1", "ID_Geno_Var2", "Gene","Cell_type","Amp_ID","ID_Geno_conc_Var1","ID_Geno_Var1_HET","ID_Geno_Var1_ALT","test","latent.var","var_type"],
    observed=True).agg({
        "p_val": "max",
        "avg_log2FC": "mean",
        "p_val_adj": "max",
        "Ncells_Var1": "sum",
        "Ncells_Var2": "sum"
}).reset_index()

sdr7_wil = pd.concat([sdr7_wil_a, sdr7_wil_b])
sdr7_wil = sdr7_wil[sdr7_wil.Gene != "DUMMY-FEATURE-FOR-SEURAT"]

sdr7_wil_pos = sdr7_wil[sdr7_wil.p_val_adj <= 0.05]
sdr7_wil_neg = sdr7_wil[(sdr7_wil.p_val_adj == 1) & (sdr7_wil.Ncells_Var1 >= 100) & (sdr7_wil.avg_log2FC.abs() <= 0.1)]

sdr7_wil_pos.loc[:,"var_type"] = "positive"
sdr7_wil_neg.loc[:,"var_type"] = "negative"

sdr7_wil = pd.concat([sdr7_wil_pos, sdr7_wil_neg])
sdr7_wil = sdr7_wil.groupby( #average sdr7a and b replicates
    ["ID_Geno_Var1", "ID_Geno_Var2", "Gene","Cell_type","Amp_ID","ID_Geno_conc_Var1","ID_Geno_Var1_HET","ID_Geno_Var1_ALT","test","latent.var","var_type"],
    observed=True).agg({
        "p_val": "max",
        "avg_log2FC": "mean",
        "p_val_adj": "max",
        "Ncells_Var1": "sum",
        "Ncells_Var2": "sum"
}).reset_index()

#takeoverlap of wilcox and mast resutls
sdr7 = pd.merge(sdr7_mast.drop(["test", "latent.var"], axis = 1), sdr7_wil.drop(["test", "latent.var"], axis = 1),
         on = ["ID_Geno_Var1", "ID_Geno_Var2", "Gene","Cell_type","Amp_ID","ID_Geno_conc_Var1","ID_Geno_Var1_HET","ID_Geno_Var1_ALT","var_type", "Ncells_Var1", "Ncells_Var2"],
         how = "inner",
         suffixes=("_MAST", "_Wilcox")
        )

In [ ]:
#sdr10
sdr10_mast = pd.read_csv(SDR010_FILE_PATH / "SDR010ABC_phased_MAST_cell_annot2_day_complement.csv")
sdr10_wil = pd.read_csv(SDR010_FILE_PATH / "SDR010ABC_phased_Wilcox_cell_annot2_day_complement.csv")

sdr10_mast = sdr10_mast[sdr10_mast.Gene != "DUMMY-FEATURE-FOR-SEURAT"]
sdr10_mast_pos = sdr10_mast[sdr10_mast.p_val_adj <= 0.05]
sdr10_mast_neg = sdr10_mast[(sdr10_mast.p_val_adj == 1) & (sdr10_mast.Ncells_Var1 >= 100) & (sdr10_mast.avg_log2FC.abs() <= 0.1)]

sdr10_mast_pos.loc[:, "var_type"] = "positive"
sdr10_mast_neg.loc[:,"var_type"] = "negative"

sdr10_mast = pd.concat([sdr10_mast_pos, sdr10_mast_neg])
sdr10_mast = sdr10_mast.groupby( #average sdr10a and b replicates
    ["ID_Geno_Var1", "ID_Geno_Var2", "Gene","Cell_type","Amp_ID","ID_Geno_conc_Var1","ID_Geno_Var1_HET","ID_Geno_Var1_ALT","test","latent.var","var_type"],
    observed=True, dropna=False).agg({
        "p_val": "max",
        "avg_log2FC": "mean",
        "p_val_adj": "max",
        "Ncells_Var1": "sum",
        "Ncells_Var2": "sum"
}).reset_index()

sdr10_wil = sdr10_wil[sdr10_wil.Gene != "DUMMY-FEATURE-FOR-SEURAT"]
sdr10_wil_pos = sdr10_wil[sdr10_wil.p_val_adj <= 0.05]
sdr10_wil_neg = sdr10_wil[(sdr10_wil.p_val_adj == 1) & (sdr10_wil.Ncells_Var1 >= 100) & (sdr10_wil.avg_log2FC.abs() <= 0.1)]

sdr10_wil_pos.loc[:, "var_type"] = "positive"
sdr10_wil_neg.loc[:,"var_type"] = "negative"

sdr10_wil = pd.concat([sdr10_wil_pos, sdr10_wil_neg])
sdr10_wil = sdr10_wil.groupby( #average sdr10a and b replicates
    ["ID_Geno_Var1", "ID_Geno_Var2", "Gene","Cell_type","Amp_ID","ID_Geno_conc_Var1","ID_Geno_Var1_HET","ID_Geno_Var1_ALT","test","latent.var","var_type"],
    observed=True, dropna=False).agg({
        "p_val": "max",
        "avg_log2FC": "mean",
        "p_val_adj": "max",
        "Ncells_Var1": "sum",
        "Ncells_Var2": "sum"
}).reset_index()

sdr10 = pd.merge(sdr10_mast.drop(["test", "latent.var"], axis = 1), sdr10_wil.drop(["test", "latent.var"], axis = 1),
         on = ["ID_Geno_Var1", "ID_Geno_Var2", "Gene","Cell_type","Amp_ID","ID_Geno_conc_Var1","ID_Geno_Var1_HET","ID_Geno_Var1_ALT","var_type", "Ncells_Var1", "Ncells_Var2"],
         how = "inner",
         suffixes=("_MAST", "_Wilcox")
        )

In [ ]:
sdr10["sdr_experiment"] = "SDR010ABC"
sdr7["sdr_experiment"] = "SDR007AB"
sdr = pd.concat([sdr7, sdr10]).reset_index(drop = True)
sdr = sdr.rename(columns={"Cell_type": "ct_day"})
sdr[["cell_type", "day"]] = sdr["ct_day"].str.split("_", expand=True)
sdr["cell_type"] = sdr.cell_type.replace("CM2", "CM").replace("FB2", "FB")
sdr = sdr.groupby( #merge eventualduplicates between sdr 7 and 10
    ["ID_Geno_Var1", "Gene","cell_type", "day", "Amp_ID","ID_Geno_conc_Var1","ID_Geno_Var1_HET","ID_Geno_Var1_ALT","var_type"],
    observed=True, dropna=False).agg({
        "p_val_MAST": "max",
        "avg_log2FC_MAST": "mean",
        "p_val_adj_MAST": "max",
        "Ncells_Var1": "sum",
        "Ncells_Var2": "sum",
        "sdr_experiment": "unique",
}).reset_index()

In [ ]:
sdr[["ID_Geno_Var1", "cell_type", "var_type", "Amp_ID"]].drop_duplicates().cell_type.value_counts()

In [ ]:
VCF = DE_to_VCF(sdr, hg38=hg38, col_to_keep = ["Gene", "cell_type", "day", "avg_log2FC_MAST", "p_val_adj_MAST", 'Ncells_Var1', "var_type", "sdr_experiment"])
VCF.head(3)

In [ ]:
VCF.loc[VCF.cell_type.isin(["CM", "FB"]), ["variant_id", "cell_type", "var_type"]].drop_duplicates().var_type.value_counts()

In [ ]:
#save vcf fiel to disk 
# VCF.to_csv(VCF_OUT_PATH, header=True, index = False)

## Metadata annotation

The annotation reads the VCF written above (`VCF_OUT_PATH`) and keeps the CM and FB variant-gene pairs.

In [ ]:
obs = pd.read_csv(VCF_OUT_PATH)
obs.rename(columns={'avg_log2FC_MAST': "logFC"}, inplace=True)
obs.shape

In [ ]:
#siplyfy columns
#obs = obs.drop(["ct_day", 'avg_log2FC.Wilcox', 'avg_log2FC.MAST', "STRAND", "sdr_experiment","day"], axis = 1).drop_duplicates()
obs = obs.drop(["STRAND", "sdr_experiment","day"], axis = 1).drop_duplicates()

#use only good celltype
obs = obs[obs.cell_type.str.contains("CM|FB")]

#drop pahsing
obs["variant_id"] = (
    obs["variant_id"]
    .str.replace("-(for)", "", regex=False)
    .str.replace("-(rev)", "", regex=False)
    .str.replace("-phased", "", regex=False)
)
obs.shape

In [ ]:
#merge days and phasing for now
obs = obs.groupby(["Gene", "variant_id", "cell_type", "var_type"], observed=True).agg({
    "logFC": "mean",
    "CHROM": "first",
    "POS": "first",
    "REF": "first",
    "ALT": "first",
}).reset_index()
obs

In [ ]:
#remove variants with ambiguous var type (results from merging days was positvie in one day and negative in the other)
duplicates = obs[obs.duplicated(subset=["Gene", "variant_id", "cell_type"], keep=False)]
duplicates

In [ ]:
obs = obs.drop_duplicates(subset=["Gene", "variant_id", "cell_type"], keep=False)

In [ ]:
#pahsed variants with identical CHROM, REF, ALT, POS but differnt log FC are still there => Aggregate
duplicates = obs[obs.duplicated(subset=["Gene", "cell_type","CHROM", "POS", "REF", "ALT"], keep=False)]
duplicates

In [ ]:
obs = obs.groupby(["Gene", "cell_type","CHROM", "POS", "REF", "ALT", "var_type"], observed=True).agg({
    "logFC": "mean",
    "variant_id": "first"
}).reset_index()
obs

In [ ]:
#here phasing lead to contractiory effects so remove
duplicates = obs[obs.duplicated(subset=["Gene", "cell_type","CHROM", "POS", "REF", "ALT"], keep=False)]
duplicates

In [ ]:
obs = obs.drop_duplicates(subset=["Gene", "cell_type","CHROM", "POS", "REF", "ALT"], keep=False)
obs.shape

In [ ]:
#add metadata annoation
#cCRE
obs_pr = vcf_to_pr(obs)
ccre_pr = load_ccre_data(ANNOT_PATH + "hg38_ENCODE4_cCREs.gz")
obs = annotate_cCREs(ccre_pr, obs_pr)

In [ ]:
#phyloP
bw_cons = pyBigWig.open(ANNOT_PATH + "hg38.phyloP100way.bw")
obs['phyloP_100way'] = annotate_phyloP_score(obs, bw_cons)
bw_cons.close()

In [ ]:
obs[['gtf_feature', 'gtf_gene_name', 'gtf_gene_type', 'closest_TSS']] = annotate_gtf_features(obs, gtf)
obs[['all_coding_effects', 'coding_effect', 'aa_change', 'impact']]  = annotate_coding_effect(obs)

In [ ]:
#annotate distance to sdr target gene TSS
sdr_transcripts = gtf.loc[(gtf.gene_name.isin(obs.Gene)) & (gtf.feature == "transcript")]
sdr_transcripts = sdr_transcripts.groupby(["chrom", "gene_name"], observed=True).agg({
    "start": "min",
    "end": "max",
    "strand": "first",
}).reset_index()

sdr_transcripts['start'] = sdr_transcripts['start'] - 1
sdr_transcripts['tss'] = np.where(sdr_transcripts['strand'] == '+', sdr_transcripts['start'], sdr_transcripts['end'] - 1)

obs = pd.merge(obs, sdr_transcripts[["chrom", "gene_name", "tss"]], how = "left", left_on=["Chromosome", "Gene"], right_on=["chrom", "gene_name"])
obs["dist_target_gene_TSS"] = obs["tss"] - obs["Pos"]
obs = obs.drop(["gene_name", "tss", "chrom"], axis = 1)

In [ ]:
#get SNV CADD scores
CADD_paths = {
    'A': ANNOT_PATH + 'CADD_snv/a.bw',
    'C': ANNOT_PATH + 'CADD_snv/c.bw',
    'G': ANNOT_PATH + 'CADD_snv/g.bw',
    'T': ANNOT_PATH + 'CADD_snv/t.bw',
}
cadd_snv = annotate_cadd_snv_scores(obs, CADD_paths)

In [ ]:
obs = pd.merge(obs, cadd_snv, on=['Chromosome', 'Pos', 'Ref', 'Alt'], how='left')

In [ ]:
CADD_indels = obs.loc[~(obs.Ref.str.len() == 1) | ~(obs.Alt.str.len() == 1),
                           ['Chromosome', 'Pos', 'variant_id', 'Ref', 'Alt']
                          ]

filename = ANNOT_PATH + "CADD_indels_res_sdr/indels_for_cadd_sdr.tsv.gz"
# CADD_indels.to_csv(filename, sep='\t', index=False, header=False, compression='gzip')

Upload to https://cadd.gs.washington.edu/score and run it. Quick note If i do this in the future: This took hours so probabaly looking up the SNVs from UCSC table browser and only submitting the indels and MNVs would be a lot faster.

In [ ]:
#read the indel prediction once their done
CADD_indels = pd.read_csv(ANNOT_PATH + "CADD_indels_res_sdr/indels_for_cadd_sdr_res.tsv.gz",
                          sep='\t',
                          comment='#',
                          compression=None,
                          names=['Chromosome', 'Pos', 'Ref', 'Alt', 'RawScore', 'CADD_PHRED_score']).drop('RawScore', axis = 1)
CADD_indels["Chromosome"] = ["chr" + str(c) for c in CADD_indels["Chromosome"]]
CADD_indels.head(5)

In [ ]:
obs = pd.merge(obs, CADD_indels, 
                        on=['Chromosome', 'Pos', 'Ref', 'Alt'], 
                        how='left',
                        suffixes=("", "_indel")
               )
obs["CADD_PHRED_score"] = obs["CADD_PHRED_score"].fillna(obs["CADD_PHRED_score_indel"])
obs.drop("CADD_PHRED_score_indel", axis=1, inplace=True)
obs

In [ ]:
#add enhacer information
enhancer_regions = pd.read_csv(ENHANCER_DIR, sep = "\t")
cts = []
for s in enhancer_regions.samples:
    d_cts = s.split(",")
    ct = {d_ct.strip().split('_', 1)[1] for d_ct in d_cts}
    if 'Fibroblasts' in ct or 'Cardiomyocytes' in ct:
        cts.append(ct)
    else:
        cts.append(set())
enhancer_regions["celltype"] = cts 

enhancer_regions[enhancer_regions.celltype.astype(bool)] #keep only non empty cell type sets 

enhancer_vars = annotate_cardioid_enhancer_overlap(variants_df = obs, regions_df = enhancer_regions)
enhancer_vars["in_cardioid_enhancer"] = True

obs = pd.merge(obs,
         enhancer_vars[["Chromosome", "Pos", "Ref","Alt", "variant_id", "enhancer_regulatory_position","is_cardioid_H3K27ac","celltypes_in_enhancer", "in_cardioid_enhancer"]], 
                        on=["Chromosome", "Pos", "Ref","Alt", "variant_id"], 
                        how='left')

obs = obs.fillna({"in_cardioid_enhancer": False})

In [ ]:
#add ATAC peak info
peak_files = glob.glob(os.path.join(PEAK_DIR, "*_peaks.narrowPeak"))
cell_types = {celltype_from_peak_file(p): p for p in peak_files if  celltype_from_peak_file(p) in ['Fibroblasts', 'Cardiomyocytes']}
print("Cell types found:", list(cell_types.keys()))

CA_variants = {}
for ct, pf in cell_types.items():
    peaks = read_narrowpeak(pf)
    mask  = flag_variants_in_peaks(obs, peaks)
    ct_results = {}
    for cohort, grp in obs[['var_type', "variant_id", "cCRE_class"]].drop_duplicates().groupby("var_type", observed=False):
        grp_mask = mask[grp.index]
        ct_results[cohort] = {
            "in":  grp[grp_mask],
            "out": grp[~grp_mask],
        }
    CA_variants[ct] = ct_results
    print(f"  {ct}: done")

In [ ]:
#get teh vairnts that show up in one CA region in at least one cell type
CA_var_ids = set.union(*[set([*res['negative']["in"]["variant_id"], *res['positive']["in"]["variant_id"]]) for res in CA_variants.values()])
obs["is_cardioid_CA"] = obs.variant_id.isin(CA_var_ids)

In [ ]:
obs = to_vcf_column_names(obs)

In [ ]:
obs.shape
# obs.to_csv(SDR_VCF_META_FILE, header=True, index=False)